# VQE for H$_2$: full pipeline from molecule to ground state energy

*Part of the QUEST Chemistry & Physics series*

Most VQE tutorials hand you a Hamiltonian and say "now run VQE on this." That skips the entire interesting first half of quantum chemistry: how you get from a molecular geometry to something you can feed into a quantum circuit. This notebook does not skip that part.

We start with the H$_2$ molecule specified only by its bond distance. We compute one-body and two-body electronic integrals, construct the second-quantized Hamiltonian, apply the Jordan-Wigner transformation to get a qubit Hamiltonian, choose a hardware-efficient ansatz, and run VQE. Then we sweep the bond distance to reproduce the H$_2$ potential energy surface (PES) and compare against Full Configuration Interaction (FCI) as the exact reference.

Finally, we run a single point of the PES on real quantum hardware to see how noise affects the estimated ground state energy.

**Learning objectives.** By the end of this notebook you will:

1. Understand the electronic structure problem and how to go from molecular geometry to a qubit Hamiltonian.
2. Implement Jordan-Wigner mapping explicitly and see how fermionic operators become Pauli strings.
3. Run VQE on an ideal simulator and reproduce the H$_2$ potential energy surface.
4. Compare VQE against Full Configuration Interaction for validation.
5. Execute a VQE evaluation on real quantum hardware and analyze the error contributions.

**What to bring in.** You should have seen at least undergraduate quantum mechanics (bra-ket notation, Hamiltonians, eigenvalue problems). Familiarity with the electronic structure problem helps but is not required; there's a brief review below.

**Credit budget.** Ansatz training on simulator is free. A single hardware energy evaluation at the optimized parameters uses approximately 1,000 shots per Pauli group × 5 groups = 5,000 shots. Total credit cost is a few dollars.


## The electronic structure problem

For a molecule with $N$ nuclei and $n$ electrons, the non-relativistic time-independent Schrodinger equation in the Born-Oppenheimer approximation is:

$$\hat{H}_{\text{elec}} |\Psi\rangle = E |\Psi\rangle$$

where $\hat{H}_{\text{elec}}$ acts only on electronic coordinates (nuclei are frozen at fixed positions). The ground state energy $E_0$ depends parametrically on the nuclear geometry. Plotting $E_0$ vs bond distance gives the *potential energy surface*, which governs molecular bonding, reaction barriers, and vibrational spectra.

For a two-electron system in a chosen basis of one-electron orbitals $\{\phi_p\}$, the second-quantized Hamiltonian is:

$$\hat{H} = \sum_{pq} h_{pq} \hat{a}^\dagger_p \hat{a}_q + \frac{1}{2}\sum_{pqrs} h_{pqrs} \hat{a}^\dagger_p \hat{a}^\dagger_q \hat{a}_r \hat{a}_s$$

where $h_{pq}$ are the one-body integrals (kinetic energy + electron-nuclear attraction) and $h_{pqrs}$ are the two-body integrals (electron-electron repulsion). PySCF computes these integrals classically given the geometry and basis set.

The fermion operators $\hat{a}^\dagger_p, \hat{a}_q$ satisfy anticommutation relations and act on a Fock space, not on qubits. To simulate this Hamiltonian on a quantum computer, we need to map fermionic operators to qubit operators (Pauli strings). The Jordan-Wigner transformation is the simplest such mapping and the one we'll use.

For H$_2$ in the minimal STO-3G basis, there are 4 spin-orbitals (2 spatial × 2 spin), giving a 4-qubit Hamiltonian with roughly 15 Pauli terms. Small enough for pedagogy, non-trivial enough to be interesting.


## Setup

In [ ]:
# Scientific Python
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Chemistry
from pyscf import gto, scf, fci
from openfermion import MolecularData, jordan_wigner
from openfermion.transforms import get_fermion_operator
from openfermionpyscf import run_pyscf

# Qiskit
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer import AerSimulator
from scipy.optimize import minimize

# qBraid
from qbraid.runtime import QbraidProvider

plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

print("Setup complete.")

## Step 1: molecular geometry and electronic integrals

We start with H$_2$ at a chosen bond distance $R$. PySCF computes the required one- and two-electron integrals in an orthonormal molecular orbital basis.

We use the STO-3G basis set: a minimal basis with one contracted Gaussian per hydrogen 1s orbital. This gives us 2 spatial orbitals (bonding + antibonding), which double to 4 spin-orbitals after accounting for spin up/down. 4 spin-orbitals means 4 qubits, which is where we want to be for pedagogy.


In [ ]:
def build_h2_hamiltonian(bond_distance):
    """
    Given H2 bond distance R in Angstroms, return the qubit Hamiltonian
    as a SparsePauliOp, and the FCI ground state energy for reference.
    """
    geometry = [('H', (0., 0., 0.)), ('H', (0., 0., bond_distance))]
    molecule = MolecularData(geometry=geometry, basis='sto-3g', multiplicity=1, charge=0)
    molecule = run_pyscf(molecule, run_scf=True, run_fci=True)

    fermion_op = get_fermion_operator(molecule.get_molecular_hamiltonian())
    qubit_op = jordan_wigner(fermion_op)

    # Convert openfermion QubitOperator -> qiskit SparsePauliOp
    pauli_list = []
    for pauli_tuple, coeff in qubit_op.terms.items():
        # pauli_tuple is like (('X', 0), ('Y', 1), ...)
        n_qubits = 4
        label = ['I'] * n_qubits
        for qubit_idx, pauli_char in pauli_tuple:
            label[qubit_idx] = pauli_char
        # Qiskit label is little-endian: reverse
        label_str = ''.join(reversed(label))
        pauli_list.append((label_str, coeff.real))

    hamiltonian = SparsePauliOp.from_list(pauli_list)
    return hamiltonian, molecule.fci_energy, molecule.hf_energy


# Build at equilibrium bond distance (H2 experimental value: ~0.74 A)
R_eq = 0.74
hamiltonian, E_fci, E_hf = build_h2_hamiltonian(R_eq)

print(f"Bond distance: {R_eq} A")
print(f"Number of qubits: {hamiltonian.num_qubits}")
print(f"Number of Pauli terms: {len(hamiltonian)}")
print(f"HF energy: {E_hf:.6f} Hartree")
print(f"FCI energy (exact): {E_fci:.6f} Hartree")
print(f"\nFirst few Pauli terms:")
for i, (p, c) in enumerate(zip(hamiltonian.paulis, hamiltonian.coeffs)):
    print(f"  {p}: {c.real:+.6f}")
    if i > 6: break

Notice the structure: an identity term (constant nuclear repulsion), single-Pauli-Z terms (diagonal), and multi-qubit Pauli terms that involve entangled measurements. Each of these terms will contribute a measurement to our energy estimator.

The FCI energy is the exact ground state energy in this basis. It's our gold standard: any VQE result must approach FCI to be considered "correct" for this basis. The HF energy is the Hartree-Fock (mean-field) energy, which is what a single Slater determinant gives you. The gap between HF and FCI, called correlation energy, is what quantum chemistry algorithms are trying to capture.


## Step 2: understanding the Jordan-Wigner transformation

The JW transformation maps fermionic creation/annihilation operators to qubit operators:

$$\hat{a}^\dagger_p = \frac{1}{2}\left(\prod_{k<p} Z_k\right)(X_p - iY_p), \quad \hat{a}_p = \frac{1}{2}\left(\prod_{k<p} Z_k\right)(X_p + iY_p)$$

The $\prod_{k<p} Z_k$ string enforces fermionic anticommutation: it tracks whether an "even" or "odd" number of electrons occupy lower-indexed orbitals. This is why single-body fermion operators can produce multi-qubit Pauli strings.

The mapping preserves all commutation and anticommutation relations, but it changes locality: nearest-neighbor fermion operators can become nonlocal qubit operators. That affects circuit depth and connectivity requirements, which we'll discuss when we look at hardware execution.

For H$_2$ specifically, the JW-mapped Hamiltonian involves qubit operators up to weight 4 (i.e., strings like $Y_0 X_1 X_2 Y_3$). These non-local terms are the ones that will suffer most from hardware noise, because implementing them requires entangling gates across multiple qubits.


## Step 3: choosing an ansatz

The next choice is the parametrized quantum circuit whose parameters we'll optimize. Two common options. UCCSD (Unitary Coupled Cluster with Singles and Doubles) is chemistry-motivated with few parameters, but the circuits run deep. It works best on ideal simulators and is often too deep for NISQ devices. A hardware-efficient ansatz (HEA) uses shallow layered circuits tailored to native gates. You give up the chemistry guarantees, but it survives on hardware.

For pedagogy and hardware execution, we use HEA with 2 layers. Each layer applies a $R_y$ rotation to every qubit followed by nearest-neighbor CNOT entanglers. Total parameter count for 2 layers on 4 qubits: $4 \times (2+1) = 12$ parameters.


In [ ]:
def hardware_efficient_ansatz(params, n_qubits=4, n_layers=2):
    """
    Layered ansatz: Ry rotations + linear CNOT entanglers, repeated n_layers times,
    followed by a final layer of Ry rotations.
    Total parameter count = n_qubits * (n_layers + 1).
    """
    qc = QuantumCircuit(n_qubits)
    idx = 0
    # Initial layer of Ry
    for q in range(n_qubits):
        qc.ry(params[idx], q)
        idx += 1
    # Repeating blocks: CNOT chain + Ry layer
    for _ in range(n_layers):
        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)
        for q in range(n_qubits):
            qc.ry(params[idx], q)
            idx += 1
    return qc


n_params = 4 * (2 + 1)
print(f"Ansatz parameter count: {n_params}")

# Draw an example
test_params = np.random.uniform(-np.pi, np.pi, n_params)
ansatz_example = hardware_efficient_ansatz(test_params)
ansatz_example.draw('mpl', fold=100)

## Step 4: VQE on the ideal simulator

VQE minimizes the expectation value $\langle \psi(\theta) | \hat{H} | \psi(\theta) \rangle$ over parameters $\theta$. We use SciPy's COBYLA optimizer, which is gradient-free and robust for noisy objective functions.

We compute the energy expectation by measuring each Pauli term separately (or in commuting groups) and combining with the appropriate coefficients.


In [ ]:
def compute_energy_ideal(params, hamiltonian):
    """Compute <psi(params)|H|psi(params)> exactly using statevector simulation."""
    qc = hardware_efficient_ansatz(params)
    sv = Statevector.from_instruction(qc)
    return np.real(sv.expectation_value(hamiltonian))


def run_vqe(hamiltonian, n_params, max_iter=200):
    """Run VQE with COBYLA optimizer."""
    history = {'energies': [], 'params': []}

    def objective(params):
        e = compute_energy_ideal(params, hamiltonian)
        history['energies'].append(e)
        history['params'].append(params.copy())
        return e

    # Random initial parameters
    x0 = np.random.uniform(-np.pi, np.pi, n_params)

    result = minimize(objective, x0, method='COBYLA',
                      options={'maxiter': max_iter, 'rhobeg': 0.5})
    return result, history


print(f"Starting VQE at bond distance {R_eq} A...")
vqe_result, history = run_vqe(hamiltonian, n_params)
print(f"Optimized energy: {vqe_result.fun:.6f} Hartree")
print(f"FCI energy:       {E_fci:.6f} Hartree")
print(f"Error:            {abs(vqe_result.fun - E_fci) * 1000:.3f} mHartree")
print(f"Iterations:       {len(history['energies'])}")

In [ ]:
# Plot convergence
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(history['energies'], color='#a02580', linewidth=1.5)
ax.axhline(E_fci, color='k', linestyle='--', alpha=0.6, label=f'FCI = {E_fci:.4f}')
ax.axhline(E_hf, color='gray', linestyle=':', alpha=0.6, label=f'HF = {E_hf:.4f}')
ax.set_xlabel('COBYLA iteration')
ax.set_ylabel('Energy (Hartree)')
ax.set_title(f'VQE convergence for H$_2$ at R = {R_eq} A')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The energy should converge to within a few mHartree of the FCI reference. "Chemical accuracy" typically means 1 mHartree (~0.6 kcal/mol), the threshold below which quantum chemistry predictions are considered reliable for many applications.

If your VQE converges to something above the HF energy, you likely got stuck in a local minimum. Restart with a new random seed. Barren plateaus and local minima plague VQE on any hardware; the field has developed initialization strategies to cope (parameter warm-starts, layerwise training), but we won't cover them here.


## Step 5: potential energy surface scan

Now we sweep bond distance and reproduce the H$_2$ dissociation curve. This is the payoff plot for any quantum chemistry method: shape of the curve, position of the minimum, and behavior at large R all diagnose the method's quality.


In [ ]:
distances = np.linspace(0.3, 2.5, 12)
vqe_energies = []
fci_energies = []
hf_energies = []

for R in distances:
    print(f"R = {R:.3f} A ...", end=' ')
    H_R, E_fci_R, E_hf_R = build_h2_hamiltonian(R)
    vqe_res, _ = run_vqe(H_R, n_params, max_iter=150)
    vqe_energies.append(vqe_res.fun)
    fci_energies.append(E_fci_R)
    hf_energies.append(E_hf_R)
    print(f"VQE={vqe_res.fun:.4f}, FCI={E_fci_R:.4f}, err={abs(vqe_res.fun-E_fci_R)*1000:.2f} mH")

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(distances, hf_energies, ':', color='gray', linewidth=2, label='Hartree-Fock')
ax.plot(distances, fci_energies, '-', color='black', linewidth=2, alpha=0.7, label='FCI (exact)')
ax.plot(distances, vqe_energies, 'o', color='#a02580', markersize=10, label='VQE (simulator)',
        markeredgecolor='white', markeredgewidth=1.5)
ax.set_xlabel('H-H bond distance (A)', fontsize=12)
ax.set_ylabel('Energy (Hartree)', fontsize=12)
ax.set_title('H$_2$ potential energy surface: VQE vs FCI vs HF', fontsize=13, pad=15)
ax.grid(alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

VQE should track FCI across the full curve, from the strongly bound region near equilibrium ($R \approx 0.74$ A) through the dissociation limit ($R \to \infty$, where H$_2$ becomes two isolated H atoms). Hartree-Fock breaks down at large R because a single Slater determinant cannot describe the diradical dissociation limit. This is where VQE (and its classical counterpart, FCI) shine: they capture the multi-reference character that HF misses.

*A physics point worth pausing on: near dissociation, the exact wavefunction is a superposition of two Slater determinants (bonding and antibonding, each with the same weight). HF forces a single determinant, so it gets the wrong energy. VQE's ansatz has enough flexibility to represent the superposition, which is why it agrees with FCI. This is the concrete pedagogical answer to "why do we need quantum computers for chemistry?"*


## Step 6: evaluate on real hardware

Full VQE parameter optimization on hardware is expensive: hundreds of iterations × many shots per iteration × noise-corrupted gradients. In practice, we optimize on the simulator (as we did above) and then evaluate the energy at the optimized parameters on hardware, comparing against the simulator value.

We'll use the optimized parameters from equilibrium ($R = 0.74$ A) and estimate the energy on a real QPU. We group the Pauli terms into commuting sets so they can be measured with fewer distinct circuit executions.


In [ ]:
# Get the optimized parameters for R=0.74
opt_params_eq = history['params'][-1]

# Retrieve the equilibrium Hamiltonian
H_eq, E_fci_eq, _ = build_h2_hamiltonian(R_eq)

# Set up qBraid device
provider = QbraidProvider()
# Instructor: replace with an available backend from your account.
DEVICE_ID = 'iqm_garnet'
device = provider.get_device(DEVICE_ID)

# For hardware execution we use qiskit's Estimator primitive style
# via manual per-term measurement. To avoid excessive shot cost,
# we group commuting Pauli terms.

def group_commuting(hamiltonian):
    """Group Pauli terms into qubit-wise commuting sets for joint measurement."""
    groups = hamiltonian.group_commuting(qubit_wise=True)
    return groups


groups = group_commuting(H_eq)
print(f"Hamiltonian has {len(H_eq)} Pauli terms, grouped into {len(groups)} commuting sets.")
for i, g in enumerate(groups):
    print(f"  Group {i+1}: {len(g)} terms")

In [ ]:
def measure_pauli_group_on_hardware(ansatz_params, pauli_group, device, shots=1024):
    """
    Given an ansatz and a commuting Pauli group, measure the expectation
    value of each term in the group on real hardware.
    Returns a dict of {pauli_label: expectation_value}.
    """
    # Build the ansatz circuit
    qc = hardware_efficient_ansatz(ansatz_params)
    n = qc.num_qubits

    # Determine measurement basis rotations from the "representative" of the group
    # For qubit-wise commuting, all terms in the group have the same Pauli letter on each qubit
    # (or identity). So we can determine per-qubit rotation from any non-identity term.
    per_qubit_basis = ['I'] * n
    for pauli in pauli_group.paulis:
        pauli_str = pauli.to_label()  # little-endian
        # Reverse to get big-endian per-qubit
        for q, ch in enumerate(pauli_str[::-1]):
            if ch != 'I':
                per_qubit_basis[q] = ch

    # Add basis rotations
    for q, basis in enumerate(per_qubit_basis):
        if basis == 'X':
            qc.h(q)
        elif basis == 'Y':
            qc.sdg(q)
            qc.h(q)
        # Z: no rotation needed

    qc.measure_all()

    # Submit to hardware
    job = device.run(qc, shots=shots)
    result = job.result()
    counts = result.data.get_counts()

    # For each Pauli term in the group, compute expectation value from counts
    expectations = {}
    total = sum(counts.values())
    for pauli, coeff in zip(pauli_group.paulis, pauli_group.coeffs):
        pauli_str = pauli.to_label()
        exp_val = 0.0
        for bitstring, count in counts.items():
            # Parity of measurement on non-identity qubits gives eigenvalue
            parity = 0
            for q, ch in enumerate(pauli_str[::-1]):
                if ch != 'I':
                    if bitstring[-(q+1)] == '1':
                        parity += 1
            sign = 1 if parity % 2 == 0 else -1
            exp_val += sign * count / total
        expectations[pauli.to_label()] = (exp_val, coeff.real)

    return expectations


# Compute the total energy from hardware measurements
print(f"Estimating H2 energy on {DEVICE_ID} at R={R_eq} A...")
total_energy_hw = 0.0
for i, group in enumerate(groups):
    print(f"  Measuring group {i+1}/{len(groups)}...")
    exps = measure_pauli_group_on_hardware(opt_params_eq, group, device, shots=1024)
    for label, (val, coeff) in exps.items():
        total_energy_hw += val * coeff

print(f"\nHardware VQE energy: {total_energy_hw:.5f} Hartree")
print(f"Simulator VQE energy: {vqe_result.fun:.5f} Hartree")
print(f"FCI energy (exact):   {E_fci_eq:.5f} Hartree")
print(f"HW-Sim error:         {abs(total_energy_hw - vqe_result.fun) * 1000:.2f} mHartree")
print(f"HW-FCI error:         {abs(total_energy_hw - E_fci_eq) * 1000:.2f} mHartree")

The hardware energy will typically be off from the simulator by 10 to 100 mHartree, well above chemical accuracy. Where does the error come from? Shot noise, first: with 1024 shots per group, individual expectation values carry statistical uncertainty of $\sim 1/\sqrt{1024} \approx 0.03$. Gate noise adds to that. Two-qubit gate errors (~1%) accumulate through the ansatz, and the hardware-efficient ansatz has multiple entangling gates, each corrupting the prepared state slightly. Readout errors (~1-3%) then bias the measured expectation values on top of everything else.

Chemical accuracy on real hardware requires either much lower noise (fault-tolerant hardware) or aggressive error mitigation (ZNE, PEC, symmetry verification, etc.). This is why VQE research today focuses heavily on error mitigation and problem-tailored ansatze.


## Discussion: what VQE teaches us about NISQ chemistry

VQE is often described as "the killer app for NISQ." That framing is optimistic. Here is what this notebook actually shows.

What works on hardware today: 4-qubit systems (H$_2$, He, LiH single-electron approximation, etc.) with careful ansatz choice and mitigation, the qualitative shape of PESs (equilibrium bond, dissociation), and comparisons against classical methods for pedagogy or benchmarking.

What doesn't work yet: chemical accuracy for anything larger than H$_2$ or He without heavy mitigation, reliable comparison of near-degenerate states (excited states, transition states), and systems with correlated multi-reference character where the ansatz has to capture subtle mixing.

The situation is not static. Two-qubit gate fidelities keep improving and qubit counts keep growing. Mitigation techniques are getting better (symmetry-based, reference-based), and symmetry-adapted ansatze avoid barren plateaus. Error correction would change the picture entirely, but for chemistry it is still years out.

The next notebook in this series explores time evolution of a many-body model (transverse-field Ising), a different application area where quantum computers may reach useful regimes sooner than for chemistry.


## Where to go next

- **Change the molecule.** Replace `H` with `He` (helium atom) or use a different basis set (`631g` instead of `sto-3g`). See how the qubit count and Pauli-term count scale.
- **Try UCCSD.** OpenFermion + Qiskit-Nature provides UCCSD ansatze. Compare convergence quality vs the hardware-efficient ansatz on the simulator.
- **Add error mitigation.** Apply zero-noise extrapolation (see the QPE notebook in the Foundations series) to the hardware energy evaluation. How much of the ideal accuracy can you recover?
- **Study a chemically interesting system.** Diatomic hydrides (LiH, BeH) are the natural next step. They're larger (up to 12 qubits) but still tractable on today's hardware for a single-point evaluation.


---

**Feedback for the QUEST pedagogy study**

Your feedback informs the IRB-approved QUEST research project on quantum computing pedagogy. Please spend 2 minutes on the following:

1. What was the most useful part of this notebook for your learning?
2. What was the most confusing or under-explained?
3. Which quantum chemistry method would you most want to see analyzed on real hardware next?

Please submit your responses via the QUEST portal or reply to your instructor.
